# Brightleaf Coffee Co. — Sales Snapshot (Python)

A tiny Python analysis for demonstrating Git in Posit Workbench (JupyterLab, VS Code, Positron).

> **This project contains synthetic data and analysis created for demonstration purposes only.**

In [ ]:
from pathlib import Path
import subprocess, sys

import polars as pl
import matplotlib.pyplot as plt

data_file = Path("data/synthetic-coffee-sales.csv")

# Synthetic data is not committed to Git (see .gitignore) — generate it if missing
if not data_file.exists():
    subprocess.run([sys.executable, "data/generate_data.py"], check=True)

sales = pl.read_csv(data_file, comment_prefix="#", try_parse_dates=True)
sales.head()

## Revenue by store

In [ ]:
(
    sales.group_by("store")
    .agg(pl.col("cups_sold").sum().alias("cups"), pl.col("revenue").sum().round(0))
    .sort("revenue", descending=True)
)

## Monthly cups sold by drink

In [ ]:
monthly = (
    sales.with_columns(pl.col("date").dt.truncate("1mo").alias("month"))
    .group_by("month", "drink")
    .agg(pl.col("cups_sold").sum().alias("cups"))
    .sort("month")
)

fig, ax = plt.subplots(figsize=(8, 4))
for drink in sorted(monthly["drink"].unique()):
    d = monthly.filter(pl.col("drink") == drink)
    ax.plot(d["month"], d["cups"], marker="o", label=drink)
ax.set_title("Cold Brew climbs as the weather warms")
ax.set_ylabel("Cups sold")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## Key takeaway

Cold Brew starts the year as the slowest seller and finishes as the top drink by June.

*Git demo tip: edit this cell, save, and watch the change appear in the Git panel.*